In [37]:
import warnings
warnings.filterwarnings('ignore')
from os.path import basename, exists


In [38]:
# Load necessary  libraries

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from scipy.stats import zscore

In [39]:
# import CSV imported from education departments website "
df = pd.read_csv('Most-Recent-Cohorts-Institution-2.csv', sep=',',low_memory=False)
df.head(5)

,UNITID,OPEID,OPEID6,INSTNM,CITY,STABBR,ZIP,ACCREDAGENCY,INSTURL,NPCURL,...,COUNT_WNE_MALE0_P11,COUNT_WNE_MALE1_P11,GT_THRESHOLD_P11,MD_EARN_WNE_INC1_P11,MD_EARN_WNE_INC2_P11,MD_EARN_WNE_INC3_P11,MD_EARN_WNE_INDEP0_P11,MD_EARN_WNE_INDEP1_P11,MD_EARN_WNE_MALE0_P11,MD_EARN_WNE_MALE1_P11
0,100654,100200.0,1002.0,Alabama A & M University,Normal,AL,35762,Southern Association of Colleges and Schools C...,www.aamu.edu/,www.aamu.edu/admissions-aid/tuition-fees/net-p...,...,800.0,777.0,0.6250,36650.0,41070.0,47016.0,38892.0,41738.0,38167.0,40250.0
1,100663,105200.0,1052.0,University of Alabama at Birmingham,Birmingham,AL,35294-0110,Southern Association of Colleges and Schools C...,https://www.uab.edu/,https://tcc.ruffalonl.com/University of Alabam...,...,1811.0,1157.0,0.7588,47182.0,51896.0,54368.0,50488.0,51505.0,46559.0,59181.0
2,100690,2503400.0,25034.0,Amridge University,Montgomery,AL,36117-3553,Southern Association of Colleges and Schools C...,https://www.amridgeuniversity.edu/,https://www2.amridgeuniversity.edu:9091/,...,75.0,67.0,0.5986,35752.0,41007.0,NaN,NaN,38467.0,32654.0,49435.0
3,100706,105500.0,1055.0,University of Alabama in Huntsville,Huntsville,AL,35899,Southern Association of Colleges and Schools C...,www.uah.edu/,finaid.uah.edu/,...,810.0,802.0,0.7810,51208.0,62219.0,62577.0,55920.0,60221.0,47787.0,67454.0
4,100724,100500.0,1005.0,Alabama State University,Montgomery,AL,36104-0271,Southern Association of Colleges and Schools C...,www.alasu.edu/,www.alasu.edu/cost-aid/tuition-costs/net-price...,...,1224.0,1049.0,0.5378,32844.0,36932.0,37966.0,34294.0,31797.0,32303.0,36964.0


In [40]:
# 1. Make headers consistent so making them all lowercase & replacing spaces with underscores 
df.columns = df.columns.str.lower().str.replace(' ', '_')

In [41]:
# Its always good to get a descriptive analystics on variables before start applying transformations
missing_cols = df.isna().sum()
missing_cols = missing_cols[missing_cols == 6484]
missing_cols

locale2            6484
ug                 6484
ugds_whitenh       6484
ugds_blacknh       6484
ugds_api           6484
                   ... 
d150_l4_whitenh    6484
d150_l4_blacknh    6484
d150_l4_api        6484
d150_l4_aianold    6484
d150_l4_hispold    6484
Length: 75, dtype: int64

In [42]:
# 2. There are 75 columns with all missing values and does not make sense as of now to keep them. 
# But lets keep the threshold at 60% to make sure our dataset is not bulky with lots of missing columns
df = df.dropna(axis=1, thresh=int(df.shape[0] * 0.60))
df.head()

,unitid,opeid,opeid6,instnm,city,stabbr,zip,accredagency,insturl,npcurl,...,count_wne_indep1_p11,count_wne_male0_p11,count_wne_male1_p11,gt_threshold_p11,md_earn_wne_inc1_p11,md_earn_wne_inc2_p11,md_earn_wne_indep0_p11,md_earn_wne_indep1_p11,md_earn_wne_male0_p11,md_earn_wne_male1_p11
0,100654,100200.0,1002.0,Alabama A & M University,Normal,AL,35762,Southern Association of Colleges and Schools C...,www.aamu.edu/,www.aamu.edu/admissions-aid/tuition-fees/net-p...,...,157.0,800.0,777.0,0.6250,36650.0,41070.0,38892.0,41738.0,38167.0,40250.0
1,100663,105200.0,1052.0,University of Alabama at Birmingham,Birmingham,AL,35294-0110,Southern Association of Colleges and Schools C...,https://www.uab.edu/,https://tcc.ruffalonl.com/University of Alabam...,...,944.0,1811.0,1157.0,0.7588,47182.0,51896.0,50488.0,51505.0,46559.0,59181.0
2,100690,2503400.0,25034.0,Amridge University,Montgomery,AL,36117-3553,Southern Association of Colleges and Schools C...,https://www.amridgeuniversity.edu/,https://www2.amridgeuniversity.edu:9091/,...,127.0,75.0,67.0,0.5986,35752.0,41007.0,NaN,38467.0,32654.0,49435.0
3,100706,105500.0,1055.0,University of Alabama in Huntsville,Huntsville,AL,35899,Southern Association of Colleges and Schools C...,www.uah.edu/,finaid.uah.edu/,...,506.0,810.0,802.0,0.7810,51208.0,62219.0,55920.0,60221.0,47787.0,67454.0
4,100724,100500.0,1005.0,Alabama State University,Montgomery,AL,36104-0271,Southern Association of Colleges and Schools C...,www.alasu.edu/,www.alasu.edu/cost-aid/tuition-costs/net-price...,...,269.0,1224.0,1049.0,0.5378,32844.0,36932.0,34294.0,31797.0,32303.0,36964.0


In [43]:
# As we removed columns with all missing values, lets impute the missing values in the remaining columns
# 3. Impute missing values with mean for numeric columns

cols = df.columns

# Bring in numeric columns
num_cols = df.select_dtypes(include=[np.number]).columns
for col in num_cols:
    df[col].fillna(df[col].mean(), inplace=True)

# select text columns by excluding numeric columns from cols
# Impute missing values with mode for text columns
text_cols = df.select_dtypes(exclude =[np.number]).columns
for col in text_cols:
    df[col].fillna(df[col].mode()[0], inplace=True)

# check if any missing values are left
missing_cols = df.isna().sum()
print(missing_cols)



unitid                    0
opeid                     0
opeid6                    0
instnm                    0
city                      0
                         ..
md_earn_wne_inc2_p11      0
md_earn_wne_indep0_p11    0
md_earn_wne_indep1_p11    0
md_earn_wne_male0_p11     0
md_earn_wne_male1_p11     0
Length: 2711, dtype: int64


In [44]:
# 4. Going to use Zip code as one of the composite keys when merging the data from API & website.
# Instead of XXXXX-YYYY to just XXXXX 

df['zip'] = df['zip'].str.split('-').str[0]
df.head(5)

,unitid,opeid,opeid6,instnm,city,stabbr,zip,accredagency,insturl,npcurl,...,count_wne_indep1_p11,count_wne_male0_p11,count_wne_male1_p11,gt_threshold_p11,md_earn_wne_inc1_p11,md_earn_wne_inc2_p11,md_earn_wne_indep0_p11,md_earn_wne_indep1_p11,md_earn_wne_male0_p11,md_earn_wne_male1_p11
0,100654,100200.0,1002.0,Alabama A & M University,Normal,AL,35762,Southern Association of Colleges and Schools C...,www.aamu.edu/,www.aamu.edu/admissions-aid/tuition-fees/net-p...,...,157.0,800.0,777.0,0.6250,36650.0,41070.0,38892.000000,41738.0,38167.0,40250.0
1,100663,105200.0,1052.0,University of Alabama at Birmingham,Birmingham,AL,35294,Southern Association of Colleges and Schools C...,https://www.uab.edu/,https://tcc.ruffalonl.com/University of Alabam...,...,944.0,1811.0,1157.0,0.7588,47182.0,51896.0,50488.000000,51505.0,46559.0,59181.0
2,100690,2503400.0,25034.0,Amridge University,Montgomery,AL,36117,Southern Association of Colleges and Schools C...,https://www.amridgeuniversity.edu/,https://www2.amridgeuniversity.edu:9091/,...,127.0,75.0,67.0,0.5986,35752.0,41007.0,41993.114652,38467.0,32654.0,49435.0
3,100706,105500.0,1055.0,University of Alabama in Huntsville,Huntsville,AL,35899,Southern Association of Colleges and Schools C...,www.uah.edu/,finaid.uah.edu/,...,506.0,810.0,802.0,0.7810,51208.0,62219.0,55920.000000,60221.0,47787.0,67454.0
4,100724,100500.0,1005.0,Alabama State University,Montgomery,AL,36104,Southern Association of Colleges and Schools C...,www.alasu.edu/,www.alasu.edu/cost-aid/tuition-costs/net-price...,...,269.0,1224.0,1049.0,0.5378,32844.0,36932.0,34294.000000,31797.0,32303.0,36964.0


In [45]:
# 5. Remove duplicate rows to avoid redundancy. 
# The drop_duplicates() method removes duplicate rows from the DataFrame. 

df = df.drop_duplicates()

In [46]:
# 6. Identify and remove outliers for the variables that are numeric Using Z-score method to identify outliers
# Z-score is a measure of how many standard deviations an element is from the mean.
# We will use the Z-score method to identify outliers in the dataset.

# Calculate Z-scores for all numerical columns
numerical_columns = df.select_dtypes(include=['float64', 'int64']).columns

z_scores = df[numerical_columns].apply(zscore)
outliers = (z_scores.abs() > 3).sum()
outliers

unitid                    148
opeid                      11
opeid6                      0
sch_deg                     0
hcm2                       40
                         ... 
md_earn_wne_inc2_p11      118
md_earn_wne_indep0_p11    103
md_earn_wne_indep1_p11    109
md_earn_wne_male0_p11     120
md_earn_wne_male1_p11     113
Length: 437, dtype: int64

In [63]:
print(df)

        unitid      opeid   opeid6  \
0       100654   100200.0   1002.0   
1       100663   105200.0   1052.0   
2       100690  2503400.0  25034.0   
3       100706   105500.0   1055.0   
4       100724   100500.0   1005.0   
...        ...        ...      ...   
6479  49178301  4270802.0  42708.0   
6480  49425001  2609404.0  26094.0   
6481  49501301  4247201.0  42472.0   
6482  49501302  4247202.0  42472.0   
6483  49664501  4285601.0  42856.0   

                                                 instnm         city stabbr  \
0                              Alabama A & M University       Normal     AL   
1                   University of Alabama at Birmingham   Birmingham     AL   
2                                    Amridge University   Montgomery     AL   
3                   University of Alabama in Huntsville   Huntsville     AL   
4                              Alabama State University   Montgomery     AL   
...                                                 ...          ... 

### 1 paragraph of the ethical implications of data wrangling specific to your datasource and the steps you completed answering the following questions:
#### a. What changes were made to the data?
#### b. Are there any legal or regulatory guidelines for your data or project topic?
#### c. What risks could be created based on the transformations done?
#### d. Did you make any assumptions in cleaning/transforming the data?
#### e. How was your data sourced / verified for credibility?
#### f. Was your data acquired in an ethical way?
#### g. How would you mitigate any of the ethical implications you have identified?

#### Ans.
##### The data wrangling process involved cleaning and transforming the dataset by standardizing column headers, 
##### removing columns with excessive missing values, imputing missing data with mean or mode, 
##### removing duplicates, and identifying outliers using Z-scores. These changes were made to ensure the 
##### dataset was usable and consistent for analysis. 
#####
##### Given the dataset originates from an educational department's website, it is likely subject to legal and regulatory guidelines
##### such as FERPA (Family Educational Rights and Privacy Act) in the U.S., which governs the privacy of student records.      
#####
##### Risks include potential misrepresentation of data due to imputation or outlier removal, 
##### which could lead to biased conclusions. Assumptions were made, such as imputing missing values 
##### with averages, which may not accurately reflect the original data distribution. 
##### The data was sourced from a credible government website, ensuring its authenticity, 
##### and no unethical practices were involved in its acquisition. To mitigate ethical concerns, 
##### transparency in documenting transformations, 
##### and ensuring compliance with relevant regulations are critical steps.
